In [1]:
# This test program aims to calculate the attenuation factors using 1000 MC sample points and for one .nxspe file.
# This program is more general in the sense it can work with concave crystals and multiple crystals.
# Multiple scattering will be ignored.

using FileIO
using GeometryBasics
using BenchmarkTools
using MeshIO
using StaticArrays
using LinearAlgebra
include(joinpath(@__DIR__, "..", "Modules", "nxspe.jl"))
using .nxspe
include(joinpath(@__DIR__, "..", "Modules", "sampling.jl"))
using .sampling
include(joinpath(@__DIR__, "..", "Modules", "gen_crystal.jl"))
using .gen_crystal

In [2]:
# Extracting the contents of the .nxspe file.

en_i, azi, pol, data, Δen = nxspe.extract(joinpath(@__DIR__, "..", "input_data/LET104215_3.7meV_1to1.nxspe"))

(3.7f0, Float32[-137.16072, -137.39026, -137.62152, -137.85446, -138.08911, -138.32544, -138.5635, -138.80327, -139.04478, -139.28812  …  41.245438, 41.48468, 41.72211, 41.957756, 42.191696, 42.42389, 42.654366, 42.883137, 43.110214, 43.335594], Float32[48.282505, 48.177177, 48.07187, 47.966606, 47.861397, 47.756275, 47.65122, 47.54627, 47.441406, 47.336624  …  131.69145, 131.58684, 131.4822, 131.3775, 131.27277, 131.16801, 131.06323, 130.95845, 130.85367, 130.74889], Float32[NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], Float32[-2.96, -2.9415, -2.923, -2.9045, -2.886, -2.8675, -2.849, -2.8305, -2.812, -2.7935  …  2.7935, 2.812, 2.8305, 2.849, 2.8675, 2.886, 2.9045, 2.923, 2.9415, 2.96])

In [3]:
# Calculating ki, n_bins, n_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = nxspe.magk_calc(en_i)
# Converting ki to a static array.
ki = SVector{3, Float32}(ki)

# Extracting the number of energy bins and number of detectors.
const n_bins = length(Δen) - 1
const n_detectors = length(azi)

98304

In [4]:
# Calculating the final neutron energy, in meV, for each energy bin.

ef_bins = nxspe.ef_calc(en_i, Δen, n_bins)

320-element Vector{Float32}:
 6.65075
 6.63225
 6.6137505
 6.59525
 6.57675
 6.5582504
 6.53975
 6.52125
 6.5027504
 6.48425
 ⋮
 0.89724994
 0.8787501
 0.86025023
 0.8417499
 0.82325006
 0.8047502
 0.7862499
 0.76775
 0.7492502

In [5]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.

kx, ky, kz = nxspe.kf_calc(ef_bins, pol, azi)

(Float32[-0.98057234 -0.9825924 … 0.9892723 0.987179; -0.97920763 -0.9812249 … 0.9878955 0.9858051; … ; -0.3331611 -0.3338474 … 0.336117 0.33540577; -0.32912266 -0.3298007 … 0.33204275 0.33134013], Float32[-0.9092696 -0.9038486 … 0.9260755 0.93142897; -0.9080041 -0.90259063 … 0.9247867 0.9301326; … ; -0.30893514 -0.30709326 … 0.31464514 0.31646404; -0.30519035 -0.30337083 … 0.31083116 0.312628], Float32[1.1921976 1.1946537 … -1.1719011 -1.1694212; 1.1905383 1.1929911 … -1.1702701 -1.1677936; … ; 0.40506324 0.40589777 … -0.3981673 -0.3973247; 0.40015325 0.40097767 … -0.39334092 -0.39250854])

In [ ]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


# Dummy multiple-crystal sample comprising 7 icospheres with 320 faces each.
stl = load(joinpath(@__DIR__, "..", "STL_Files/7_Icospheres320.stl"))
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

2240

In [7]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = sampling.ve_calc(vertices, indices)

(SVector{3, Float32}[[-0.15378767, 0.95625895, -0.24883369], [-0.43765807, -0.86402386, -0.24883369], [0.15378767, 0.95625895, 0.24883369], [-0.8134424, 0.29252154, 0.5027351], [0.4861489, -0.7147844, 0.5027351], [-0.8134424, -0.29252154, 0.5027351], [0.52957207, 0.6832356, -0.5027351], [-0.43765807, 0.86402386, -0.24883369], [-0.078459844, 0.24147457, -0.9672301], [-0.23224752, -0.7147844, -0.65965474]  …  [0.8300286, 0.24147457, -2.4972649], [-0.60803187, 0.44176102, -2.3403451], [-0.35222673, 0.5877517, -2.2716565], [-0.60803187, 0.44176102, -2.3403451], [0.078459844, 0.24147457, -2.03277], [-0.14114168, 0.43438944, -2.1104019], [0.078459844, 0.24147457, -2.03277], [0.02683698, 0.86402386, -2.4972649], [-0.060514368, 0.6825348, -2.2716565], [0.02683698, 0.86402386, -2.4972649]], SVector{3, Float32}[[-0.15521169, -0.0052567124, 0.25951764], [0.12865871, -0.086978376, 0.25951764], [0.15521169, -0.0052567124, -0.25951764], [-0.13667011, -0.13915926, -0.23113567], [0.19236422, 0.0322495

In [8]:
# Setting the desired number of MC sample points and creating vector for coordinates.

const n_mc = 10
mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)

10-element Vector{SVector{3, Float32}}:
 [1.0606121f-37, 7.7f-43, 1.0559593f-37]
 [7.7f-43, 1.1256252f-37, 7.7f-43]
 [1.3217531f-37, 7.7f-43, 1.0559629f-37]
 [7.7f-43, 0.0, 0.0]
 [0.0, 0.0, 1.5380768f7]
 [4.591f-41, 1.2667003f-37, 7.7f-43]
 [1.0448171f-37, 7.7f-43, 1.0448207f-37]
 [7.7f-43, 1.0448243f-37, 7.7f-43]
 [1.0448279f-37, 7.7f-43, 1.0448315f-37]
 [7.7f-43, 1.0939993f-37, 7.7f-43]

In [9]:
# Setting the (estimated) parameters of the sample.

# The reference attenuation coefficent at 25.3 meV in cm^-1.
const μ_ref = Float32(1)
const en_ref = Float32(25.3)

25.3f0

In [10]:
# Calculating the pre-scattering attenuation coefficient.

const μi = gen_crystal.μ_calc(μ_ref, en_ref, en_i)

2.6149259f0

In [11]:
# Pre-allocating the pre-scattering path length stores.

len_i = Vector{Float32}(undef, n_mc)

10-element Vector{Float32}:
 2.2f-44
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [12]:
# Generating the sample points and the pre-scattering path lengths.

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = sampling.aabb_3d(vertices)
sampling.sample!(ranges, e2s, e3s, v1s, len_i, mc_coords, true, n_faces, n_mc)

(Float32[1.8336948, 0.77606994, 1.4800308, 1.6177194, 0.4890682, 0.8205955, 0.27950707, 5.4206405, 2.7990866, 3.4721951], SVector{3, Float32}[[0.88476527, -3.073559, 0.27462196], [0.3204172, 0.839373, 3.2821035], [0.49969184, -2.9509315, -0.10773102], [0.3509026, -0.71101385, -0.54370666], [-0.22937253, -0.5874117, -3.3319242], [-2.9231865, 0.6455183, -0.06323316], [-0.33627102, -2.6052582, 0.67237186], [3.5925634, 0.14970864, 0.122568496], [0.31636134, -0.16742015, 0.5160789], [2.4910126, 0.53090376, -0.24463883]])

In [17]:
# Testing the time taken to output this grid of attenuation factors.

atten_grid, acc_rate = gen_crystal.a_grid_calc(data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i, n_bins, n_detectors, n_faces, n_mc, μ_ref, en_ref, μi)
# Testing the same (known to be non-zero) datapoint.
display(atten_grid[160,6])
display(acc_rate)

0.03547687f0

0.99999905f0

In [14]:
# Benchmarking len_calc() with the first final wavevector and the first MC sampling point.


d_test = SVector{3, Float32}(kx[1][1], ky[1][1], kz[1][1])
d_test = d_test / norm(d_test)
λs = Vector{Float32}(undef, 10)
path_lengths = Vector{Float32}(undef, 10)
p_test = Vector{SVector{3, Float32}}(undef, n_faces)
det_test = Vector{Float32}(undef, n_faces)
gen_crystal.pdet_calc!(d_test, e2s, e3s, p_test, det_test, n_faces)
test = mc_coords[1]


@benchmark gen_crystal.len_calc(e2s, e3s, d_test, p_test, det_test, test, v1s, λs, path_lengths, n_faces)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   9.800 μs … 512.000 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     10.400 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   12.096 μs ±   6.735 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▇█▆▄▂▁▁                 ▂▅▂                             ▁▁   ▁
  █████████▇▇▆▆▅▄▅▅▅▅▄▄▅▄▇███▇▇▆▆▇▆▅▄▅▄▃▃▃▂▃▃▃▃▂▃▂▃▄▄▆▅▇█████▇ █
  9.8 μs        Histogram: log(frequency) by time      28.5 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [15]:
# Benchmarking atten_calc() for the first final wavevector and first final energy.


d_test = SVector{3, Float32}(kx[1][1], ky[1][1], kz[1][1])
d_test = d_test / norm(d_test)
λs = Vector{Float32}(undef, 10)
path_lengths = Vector{Float32}(undef, 10)
p_test = Vector{SVector{3, Float32}}(undef, n_faces)
det_test = Vector{Float32}(undef, n_faces)
gen_crystal.pdet_calc!(d_test, e2s, e3s, p_test, det_test, n_faces)
en_test = ef_bins[1]


@benchmark gen_crystal.atten_calc(d_test, en_test, v1s, e2s, e3s, mc_coords, len_i, p_test, det_test, λs, path_lengths, n_faces, n_mc, μ_ref, en_ref, μi)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  222.700 μs … 887.300 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     247.300 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   258.468 μs ±  37.983 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▇█▆▅▂▃▅▂ ▁                                                   
  ▁▅███████████▅▇▇▅▄▄▃▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▂▃▃▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁ ▃
  223 μs           Histogram: frequency by time          396 μs <

 Memory estimate: 48 bytes, allocs estimate: 2.

In [16]:
# Benchmarking a_grid_calc().


@benchmark gen_crystal.a_grid_calc(data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i, n_bins, n_detectors, n_faces, n_mc, μ_ref, en_ref, μi)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 87.233 s (0.00% GC) to evaluate,
 with a memory estimate of 143.18 MiB, over 635719 allocations.